# 03. ReAct: Reasoning + Acting

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 75 minutos  
**Prerequisitos:** [01. Intro LLM Agents](01-intro-llm-agents.ipynb), [02. Prompting Agéntico](02-prompting-agentico.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Comprender el patrón ReAct (Reasoning + Acting) y su importancia
- Implementar la alternancia explícita entre razonamiento y acción
- Construir un agente de Q&A con búsqueda en Wikipedia
- Analizar trazas de ejecución para debugging y mejora
- Comparar ReAct con otros paradigmas de agentes

## 1. Motivación: ¿Por qué ReAct?

### El Problema: Agentes Que Actúan Sin Pensar (o Piensan Sin Actuar)

**Pregunta:** *"¿En qué año nació el autor de 'Cien años de soledad' y cuántos años tenía cuando publicó ese libro?"*

**Agente sin ReAct (solo actúa):**
```
Action: search[Cien años de soledad]
Action: search[Gabriel García Márquez nacimiento]
Action: search[publicación Cien años de soledad]
...
```
❌ Búsquedas sin estrategia, difícil de seguir el razonamiento

**Agente con ReAct:**
```
Thought: Necesito saber quién escribió 'Cien años de soledad'
Action: search[autor Cien años de soledad]
Observation: Gabriel García Márquez

Thought: Ahora necesito su año de nacimiento
Action: search[Gabriel García Márquez nacimiento]
Observation: 1927

Thought: Y el año de publicación del libro
Action: search[Cien años de soledad año publicación]
Observation: 1967

Thought: Puedo calcular la edad: 1967 - 1927 = 40 años
Answer: Gabriel García Márquez nació en 1927 y tenía 40 años cuando publicó 'Cien años de soledad' en 1967.
```
✅ Razonamiento explícito, trazable, estratégico

### La Solución: ReAct Pattern

ReAct alterna sistemáticamente entre:
1. **Reasoning (Thought)**: Razonar sobre qué hacer
2. **Acting (Action)**: Ejecutar acción en el mundo
3. **Observing**: Recibir resultado
4. ↺ Repetir hasta resolver

### Pregunta Guía

**Al final responderemos:**
*¿Cómo hace que un agente sea más confiable y debuggeable la alternancia explícita entre razonamiento y acción?*

## 2. Intuición Visual: El Loop ReAct

### Arquitectura ReAct

```
┌────────────────────────────────────────────────────────────┐
│                    ReAct Loop                              │
├────────────────────────────────────────────────────────────┤
│                                                            │
│  Query ────────────────────┐                              │
│                            ↓                              │
│          ┌─────────────────────────────┐                  │
│          │                             │                  │
│          ↓                             │                  │
│    ┌──────────┐                        │                  │
│    │ Thought  │  "Necesito buscar X"   │                  │
│    └──────────┘                        │                  │
│          │                             │                  │
│          ↓                             │                  │
│    ┌──────────┐                        │                  │
│    │ Action   │  search[X]             │                  │
│    └──────────┘                        │                  │
│          │                             │                  │
│          ↓                             │                  │
│    ┌───────────┐                       │                  │
│    │Observation│  [Result]             │                  │
│    └───────────┘                       │                  │
│          │                             │                  │
│          └─────────────────────────────┘                  │
│                       │                                    │
│                       ↓                                    │
│              ¿Suficiente info?                             │
│               /           \                               │
│             Sí            No → loop again                  │
│              ↓                                             │
│         ┌────────┐                                         │
│         │ Answer │                                         │
│         └────────┘                                         │
│                                                            │
└────────────────────────────────────────────────────────────┘
```

### Ventajas de ReAct

| Aspecto | Sin ReAct | Con ReAct |
|---------|-----------|----------|
| **Trazabilidad** | ❌ Difícil seguir decisiones | ✅ Razonamiento explícito |
| **Debugging** | ❌ Black box | ✅ Cada paso es visible |
| **Confianza** | ❌ No sabes por qué actuó | ✅ Justifica acciones |
| **Error handling** | ❌ Loops sin salida | ✅ Puede autocorregirse |
| **Interpretabilidad** | ❌ Opaco | ✅ Humano puede seguirlo |

### Paper Original

**"ReAct: Synergizing Reasoning and Acting in Language Models"** (Yao et al., 2022)  
[https://arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)

**Key Finding:** Alternar razonamiento y acción mejora performance vs solo actuar o solo razonar.

In [ ]:
# Instalación de dependencias
# !pip install openai wikipedia-api python-dotenv plotly

import os
import re
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from enum import Enum

import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible")

try:
    import wikipediaapi
    WIKIPEDIA_AVAILABLE = True
except ImportError:
    WIKIPEDIA_AVAILABLE = False
    print("⚠️  Wikipedia-API no disponible. Instala: pip install wikipedia-api")

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 3. Fundamentos Matemáticos: Formalización de ReAct

### Formalización del Loop

Sea $\mathcal{T}$ el espacio de thoughts (pensamientos) y $\mathcal{A}$ el espacio de acciones:

$$
\begin{align}
\text{Trajectory: } \tau &= (t_1, a_1, o_1, t_2, a_2, o_2, ..., t_n, a_n, o_n) \tag{1} \\
\text{donde: } & \\
t_i &\in \mathcal{T}: \text{thought en paso } i \\
a_i &\in \mathcal{A}: \text{acción en paso } i \\
o_i &\in \mathcal{O}: \text{observación en paso } i
\end{align}
$$

### Política ReAct

$$
\begin{align}
t_i &= \text{LLM}_{\text{think}}(q, \tau_{<i}) \tag{2} \\
a_i &= \text{LLM}_{\text{act}}(q, \tau_{\leq i}) \tag{3} \\
o_i &= \text{Env}(a_i) \tag{4}
\end{align}
$$

Donde:
- $q$: Query inicial del usuario
- $\tau_{<i}$: Historia hasta (pero sin incluir) paso $i$
- $\text{Env}$: Entorno (herramientas, APIs, etc.)

### Criterio de Parada

El agente se detiene cuando:

$$
\begin{align}
\text{stop} &= \begin{cases}
\text{True} & \text{si } t_i = \text{"Finish[answer]"} \\
\text{True} & \text{si } i > \text{max\_steps} \\
\text{False} & \text{en otro caso}
\end{cases} \tag{5}
\end{align}
$$

### Ventaja sobre Act-Only

**Act-Only trajectory:** $(a_1, o_1, a_2, o_2, ...)$

**ReAct trajectory:** $(t_1, a_1, o_1, t_2, a_2, o_2, ...)$

El thought $t_i$ permite al LLM:
1. **Planificar** antes de actuar
2. **Reflexionar** sobre observaciones
3. **Autocorregirse** si detecta errores
4. **Mantener coherencia** en estrategia multi-paso

**Resultado empírico (paper original):**
- Act-only: ~35% success en HotpotQA
- ReAct: ~58% success en HotpotQA
- Mejora de ~65% relativa

## 4. Implementación Desde Cero: ReAct Agent

### 4.1 Definir Herramientas

In [ ]:
class WikipediaTool:
    """
    Herramienta para buscar en Wikipedia.
    """
    
    def __init__(self):
        if WIKIPEDIA_AVAILABLE:
            self.wiki = wikipediaapi.Wikipedia(
                language='es',
                user_agent='ReActAgent/1.0'
            )
        else:
            self.wiki = None
    
    def search(self, query: str, sentences: int = 3) -> str:
        """
        Busca en Wikipedia y retorna las primeras N oraciones.
        """
        if not self.wiki:
            # Fallback simulado
            return self._simulated_search(query)
        
        page = self.wiki.page(query)
        
        if not page.exists():
            return f"No se encontró página para: {query}"
        
        # Extraer primeras N oraciones
        summary = page.summary
        sentences_list = summary.split('. ')[:sentences]
        
        return '. '.join(sentences_list) + '.'
    
    def _simulated_search(self, query: str) -> str:
        """Simulación para cuando Wikipedia no está disponible"""
        simulated_data = {
            "gabriel garcía márquez": "Gabriel García Márquez (1927-2014) fue un escritor colombiano. Ganó el Premio Nobel de Literatura en 1982. Es conocido principalmente por su novela 'Cien años de soledad' publicada en 1967.",
            "cien años de soledad": "Cien años de soledad es una novela del escritor colombiano Gabriel García Márquez. Fue publicada en 1967. Es considerada una obra maestra de la literatura hispanoamericana.",
            "default": f"Información sobre {query}: [Simulado - instala wikipedia-api para búsquedas reales]"
        }
        
        query_lower = query.lower()
        for key in simulated_data:
            if key in query_lower:
                return simulated_data[key]
        
        return simulated_data["default"]

# Crear herramienta
wiki_tool = WikipediaTool()
print("✅ WikipediaTool creada")

# Probar
result = wiki_tool.search("Gabriel García Márquez")
print(f"\nEjemplo búsqueda:\n{result}")

### 4.2 Implementar ReAct Agent

In [ ]:
class StepType(Enum):
    """Tipos de pasos en ReAct"""
    THOUGHT = "thought"
    ACTION = "action"
    OBSERVATION = "observation"
    ANSWER = "answer"

@dataclass
class ReActStep:
    """Representa un paso en la trayectoria ReAct"""
    step_type: StepType
    content: str
    step_number: int

class ReActAgent:
    """
    Implementación del patrón ReAct.
    
    Paper: "ReAct: Synergizing Reasoning and Acting in Language Models"
    Yao et al., 2022
    """
    
    def __init__(self, wiki_tool: WikipediaTool, llm_backend: str = "simulated"):
        self.wiki = wiki_tool
        self.llm_backend = llm_backend
        self.trajectory: List[ReActStep] = []
        
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        else:
            self.client = None
    
    def _build_prompt(self, question: str) -> str:
        """
        Construye el prompt ReAct con ejemplos few-shot.
        """
        # Few-shot examples del paper original
        examples = """
Ejemplo 1:
Pregunta: ¿En qué año nació el autor de "El amor en los tiempos del cólera"?

Thought 1: Necesito saber quién escribió "El amor en los tiempos del cólera"
Action 1: Search[El amor en los tiempos del cólera]
Observation 1: El amor en los tiempos del cólera es una novela de Gabriel García Márquez publicada en 1985.

Thought 2: El autor es Gabriel García Márquez. Ahora necesito saber su año de nacimiento.
Action 2: Search[Gabriel García Márquez]
Observation 2: Gabriel García Márquez (1927-2014) fue un escritor colombiano.

Thought 3: Gabriel García Márquez nació en 1927.
Action 3: Finish[1927]

---
"""
        
        # Construir historial de trajectory
        history = ""
        for step in self.trajectory:
            if step.step_type == StepType.THOUGHT:
                history += f"\nThought {step.step_number}: {step.content}"
            elif step.step_type == StepType.ACTION:
                history += f"\nAction {step.step_number}: {step.content}"
            elif step.step_type == StepType.OBSERVATION:
                history += f"\nObservation {step.step_number}: {step.content}"
        
        prompt = f"""Responde preguntas usando el patrón ReAct: alternancia entre Thought, Action, Observation.

Tienes acceso a estas acciones:
- Search[query]: Busca información en Wikipedia
- Finish[answer]: Retorna la respuesta final

{examples}

Ahora responde esta pregunta:
Pregunta: {question}
{history}

Próximo paso (Thought o Action):"""
        
        return prompt
    
    def _call_llm(self, prompt: str) -> str:
        """Llama al LLM"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=200
            )
            return response.choices[0].message.content
        else:
            return self._simulated_llm(prompt)
    
    def _simulated_llm(self, prompt: str) -> str:
        """
        LLM simulado que sigue el patrón ReAct.
        """
        # Analizar estado actual
        last_step = self.trajectory[-1] if self.trajectory else None
        step_num = len([s for s in self.trajectory if s.step_type == StepType.THOUGHT]) + 1
        
        # Si no hay pasos, empezar con Thought
        if not last_step or last_step.step_type == StepType.OBSERVATION:
            # Decidir siguiente thought basado en query
            if "autor" in prompt.lower() and "cien años" in prompt.lower():
                if step_num == 1:
                    return "Thought 1: Necesito buscar quién escribió 'Cien años de soledad'"
                elif step_num == 2:
                    return "Thought 2: El autor es Gabriel García Márquez. Ahora necesito su año de nacimiento."
                elif step_num == 3:
                    return "Thought 3: García Márquez nació en 1927 y el libro se publicó en 1967. Tenía 40 años."
            return f"Thought {step_num}: Necesito más información"
        
        # Si último fue Thought, generar Action
        elif last_step.step_type == StepType.THOUGHT:
            if "buscar" in last_step.content.lower() or "necesito" in last_step.content.lower():
                if "cien años" in last_step.content.lower():
                    return f"Action {step_num}: Search[Cien años de soledad]"
                elif "garcía márquez" in last_step.content.lower() or "nacimiento" in last_step.content.lower():
                    return f"Action {step_num}: Search[Gabriel García Márquez]"
            # Finalizar
            return f"Action {step_num}: Finish[Gabriel García Márquez nació en 1927 y tenía 40 años cuando publicó 'Cien años de soledad' en 1967]"
        
        return "Thought: Continuando..."
    
    def _parse_step(self, response: str) -> Tuple[StepType, str, int]:
        """
        Parsea la respuesta del LLM para extraer tipo y contenido.
        """
        # Buscar patrones
        thought_match = re.match(r'Thought (\d+): (.+)', response, re.IGNORECASE)
        action_match = re.match(r'Action (\d+): (.+)', response, re.IGNORECASE)
        
        if thought_match:
            return StepType.THOUGHT, thought_match.group(2).strip(), int(thought_match.group(1))
        elif action_match:
            return StepType.ACTION, action_match.group(2).strip(), int(action_match.group(1))
        
        # Default
        return StepType.THOUGHT, response.strip(), len(self.trajectory) + 1
    
    def _execute_action(self, action: str) -> str:
        """
        Ejecuta una acción y retorna la observación.
        """
        # Parse action
        search_match = re.match(r'Search\[(.+)\]', action, re.IGNORECASE)
        finish_match = re.match(r'Finish\[(.+)\]', action, re.IGNORECASE)
        
        if search_match:
            query = search_match.group(1)
            return self.wiki.search(query)
        elif finish_match:
            return finish_match.group(1)
        else:
            return f"Error: Acción no reconocida: {action}"
    
    def run(
        self, 
        question: str, 
        max_steps: int = 10,
        verbose: bool = True
    ) -> Dict:
        """
        Ejecuta el agente ReAct.
        """
        if verbose:
            print(f"\n{'='*70}")
            print(f"🤖 ReAct Agent")
            print(f"{'='*70}")
            print(f"\n❓ Pregunta: {question}\n")
        
        self.trajectory = []
        
        for i in range(max_steps):
            # Construir prompt
            prompt = self._build_prompt(question)
            
            # Llamar LLM
            response = self._call_llm(prompt)
            
            # Parsear step
            step_type, content, step_num = self._parse_step(response)
            
            # Agregar a trajectory
            step = ReActStep(step_type, content, step_num)
            self.trajectory.append(step)
            
            if verbose:
                if step_type == StepType.THOUGHT:
                    print(f"💭 Thought {step_num}: {content}")
                elif step_type == StepType.ACTION:
                    print(f"🔧 Action {step_num}: {content}")
            
            # Si es acción, ejecutar
            if step_type == StepType.ACTION:
                # Verificar si es Finish
                if content.startswith("Finish["):
                    answer = re.match(r'Finish\[(.+)\]', content).group(1)
                    if verbose:
                        print(f"\n✅ Respuesta Final: {answer}")
                    return {
                        "answer": answer,
                        "trajectory": self.trajectory,
                        "steps": len(self.trajectory)
                    }
                
                # Ejecutar acción
                observation = self._execute_action(content)
                obs_step = ReActStep(StepType.OBSERVATION, observation, step_num)
                self.trajectory.append(obs_step)
                
                if verbose:
                    print(f"📊 Observation {step_num}: {observation}\n")
        
        # Max steps alcanzado
        if verbose:
            print(f"\n⚠️  Max steps ({max_steps}) alcanzado sin respuesta final")
        
        return {
            "answer": "No se pudo completar en el límite de pasos",
            "trajectory": self.trajectory,
            "steps": len(self.trajectory)
        }

print("✅ ReActAgent implementado")

### 4.3 Probar el Agente

In [ ]:
# Crear agente
agent = ReActAgent(wiki_tool, llm_backend="simulated")

# Pregunta compleja que requiere múltiples pasos
question = "¿En qué año nació el autor de 'Cien años de soledad' y cuántos años tenía cuando publicó ese libro?"

# Ejecutar
result = agent.run(question, verbose=True)

print(f"\n{'='*70}")
print(f"📈 Estadísticas:")
print(f"  - Total de pasos: {result['steps']}")
print(f"  - Respuesta: {result['answer']}")

## 5. Versión con Framework: LangChain ReAct

LangChain tiene soporte built-in para ReAct.

In [ ]:
# Ejemplo conceptual con LangChain
try:
    from langchain.agents import initialize_agent, Tool, AgentType
    from langchain_openai import ChatOpenAI
    
    # Definir herramientas
    tools = [
        Tool(
            name="Wikipedia",
            func=wiki_tool.search,
            description="Busca información en Wikipedia. Input: query de búsqueda."
        )
    ]
    
    # LLM
    # llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    
    # Agente ReAct
    # react_agent = initialize_agent(
    #     tools=tools,
    #     llm=llm,
    #     agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    #     verbose=True
    # )
    
    # result = react_agent.run(question)
    
    print("✅ LangChain ReAct disponible")
except ImportError:
    print("⚠️  LangChain no disponible")

## 6. Visualización de Trajectory

In [ ]:
def visualize_react_trajectory(trajectory: List[ReActStep]):
    """
    Visualiza la trayectoria ReAct como diagrama de flujo.
    """
    fig = go.Figure()
    
    # Colores por tipo
    colors = {
        StepType.THOUGHT: '#9b59b6',
        StepType.ACTION: '#e74c3c',
        StepType.OBSERVATION: '#2ecc71',
        StepType.ANSWER: '#f39c12'
    }
    
    # Crear nodos
    x_positions = []
    y_positions = []
    node_colors = []
    node_texts = []
    hover_texts = []
    
    for i, step in enumerate(trajectory):
        x_positions.append(i)
        y_positions.append(0)
        node_colors.append(colors[step.step_type])
        node_texts.append(step.step_type.value.upper()[:1])
        hover_texts.append(f"{step.step_type.value.upper()} {step.step_number}\n{step.content[:100]}...")
    
    # Agregar nodos
    fig.add_trace(go.Scatter(
        x=x_positions,
        y=y_positions,
        mode='markers+text',
        marker=dict(
            size=50,
            color=node_colors,
            line=dict(width=2, color='white')
        ),
        text=node_texts,
        textposition="middle center",
        textfont=dict(size=14, color='white'),
        hovertext=hover_texts,
        hoverinfo='text',
        showlegend=False
    ))
    
    # Agregar flechas
    for i in range(len(trajectory) - 1):
        fig.add_annotation(
            x=x_positions[i+1],
            y=0,
            ax=x_positions[i],
            ay=0,
            xref='x',
            yref='y',
            axref='x',
            ayref='y',
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='#34495e'
        )
    
    fig.update_layout(
        title="ReAct Trajectory",
        xaxis=dict(showticklabels=False, showgrid=False),
        yaxis=dict(showticklabels=False, showgrid=False, range=[-1, 1]),
        height=250,
        template='plotly_white',
        hovermode='closest'
    )
    
    return fig

# Visualizar
if result['trajectory']:
    fig = visualize_react_trajectory(result['trajectory'])
    fig.show()

## 7. Ejercicios

### 🟢 Ejercicio 1: Agregar Herramienta de Cálculo

In [ ]:
def ejercicio_1_calculator_tool():
    """
    Objetivo: Extender ReActAgent con una herramienta de calculadora
    
    Instrucciones:
    1. Crea una clase CalculatorTool similar a WikipediaTool
    2. Modifica ReActAgent para aceptar múltiples herramientas
    3. Actualiza _execute_action para manejar Calculator[expr]
    4. Prueba con: "¿Cuál es la raíz cuadrada de 144 más 25?"
    """
    # TODO: Tu código aquí
    pass

# ejercicio_1_calculator_tool()

### 🟡 Ejercicio 2: Self-Reflection

Implementa self-reflection: el agente revisa su trajectory y decide si cometió errores.

In [ ]:
def ejercicio_2_self_reflection():
    """
    Objetivo: Agregar capacidad de autocorrección al agente
    
    Idea:
    Después de cada N pasos, el agente revisa su trajectory:
    - "¿He progresado hacia la respuesta?"
    - "¿Estoy repitiendo búsquedas?"
    - "¿Necesito cambiar de estrategia?"
    
    Instrucciones:
    1. Agrega método _reflect() que analiza trajectory
    2. LLM evalúa si está progresando
    3. Si no, sugiere nueva estrategia
    """
    # TODO: Tu código aquí
    pass

# ejercicio_2_self_reflection()

### 🔴 Ejercicio 3: ReAct con Tree-of-Thought

Combina ReAct con ToT: en cada Thought, explorar múltiples caminos.

In [ ]:
def ejercicio_3_react_tot():
    """
    Objetivo: Integrar Tree-of-Thought en ReAct
    
    Idea:
    En lugar de un solo Thought → Action, generar:
    - Múltiples thoughts posibles
    - Evaluar cada uno
    - Seleccionar mejor
    - Ejecutar action correspondiente
    
    Desafío avanzado - requiere investigación adicional
    """
    # TODO: Tu código aquí
    pass

# Este es un ejercicio de investigación - explora papers recientes

## 8. Resumen y Recursos

### 📚 Resumen

- **ReAct Pattern**: Alternancia explícita Thought → Action → Observation
- **Ventajas clave**:
  - ✅ Trazabilidad completa de razonamiento
  - ✅ Debugging más fácil
  - ✅ Mejor performance que act-only o reason-only
  - ✅ Autocorrección posible
  
- **Componentes**:
  - Thoughts: Razonamiento explícito
  - Actions: Ejecución de herramientas
  - Observations: Resultados del entorno
  
- **Implementación**:
  - Few-shot prompting para enseñar formato
  - Parsing estructurado de outputs
  - Loop con límite de pasos
  
- **Aplicaciones**:
  - Question answering con búsqueda
  - Task automation
  - Research assistants
  - Code generation agents

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

1. **"ReAct: Synergizing Reasoning and Acting in Language Models"** (Yao et al., 2022)
   - [https://arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)
   - Paper original que introduce ReAct
   - Resultados en HotpotQA, FEVER, WebShop
   
2. **"Reflexion: Language Agents with Verbal Reinforcement Learning"** (Shinn et al., 2023)
   - [https://arxiv.org/abs/2303.11366](https://arxiv.org/abs/2303.11366)
   - Extiende ReAct con self-reflection

#### 💻 Implementaciones

- **LangChain ReAct Agent**: [Docs](https://python.langchain.com/docs/modules/agents/agent_types/react)
- **Original ReAct Code**: [GitHub](https://github.com/ysymyth/ReAct)

### ➡️ Próximo Paso

En el siguiente notebook **"04. Tool Use y Function Calling"**, profundizaremos en:

- Definición formal de herramientas
- Function calling APIs (OpenAI, Anthropic)
- Tool selection strategies
- Integración con APIs externas
- Sandboxing y seguridad
- Composición de herramientas

**[➡️ Ir al Notebook 04: Tool Use](04-tool-use-function-calling.ipynb)**

---

<div align="center">

### Respuesta a la Pregunta Guía

*¿Por qué alternar razonamiento y acción es mejor?*

**Respuesta:**
1. **Trazabilidad**: Cada decisión tiene justificación explícita
2. **Debugging**: Podemos ver exactamente dónde falló el razonamiento
3. **Coherencia**: El agente mantiene estrategia consistente multi-paso
4. **Autocorrección**: Puede detectar errores en observaciones
5. **Performance**: ~65% mejora vs act-only (paper original)

ReAct es el patrón fundamental para agentes confiables.

</div>